# Engine: BAO Mass Gap — the Yang-Mills gap

**File:** `ValaQuenta/bao_mass_gap.py`
**Wiki:** [wiki/bao_mass_gap.md](../../wiki/bao_mass_gap.md)

Two constants, one operation, one result:

```
GAP = OMEGA_ZS - D_STAR x ln(10)
```

`OMEGA_ZS` is W(1), the Lambert W function at 1 — an exact special-function
value. `D_STAR` is the Berry-Keating spectral floor, quoted to 5 significant
figures. Everything below is computed from those two numbers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, cmath
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('python', sys.version.split()[0])

In [ ]:
from ValaQuenta import bao_mass_gap as bmg

print(f'OMEGA_ZS = {bmg.OMEGA_ZS!r}')
print(f'D_STAR   = {bmg.D_STAR!r}')
print(f'LN10     = {bmg.LN10!r}')
print(f'GAP      = {bmg.GAP!r}')
print()
# Recompute from the definition rather than trusting the stored constant.
gap_recomputed = bmg.OMEGA_ZS - bmg.D_STAR * bmg.LN10
print(f'recomputed        = {gap_recomputed!r}')
print(f'matches GAP       = {gap_recomputed == bmg.GAP}')

## The identity, stated carefully

The README is emphatic about this and it is worth repeating, because getting it
wrong is the single easiest way to misread the whole engine:

```
1/sqrt(2000)      = 0.022360...   <- NOT the gap (31.6x too large)
1/(1000*sqrt(2))  = 0.000707...   <- the approximate identity
1/sqrt(2000000)   = 0.000707...   <- same thing, unambiguous
```

In [ ]:
candidates = {
    '1/sqrt(2000)':     1/math.sqrt(2000),
    '1/(1000*sqrt(2))': 1/(1000*math.sqrt(2)),
    '1/sqrt(2e6)':      1/math.sqrt(2_000_000),
}
print(f'{"expression":20s} {"value":>16s} {"ratio to GAP":>14s} {"rel. error":>12s}')
for name, v in candidates.items():
    print(f'{name:20s} {v:16.12f} {v/bmg.GAP:14.4f} {abs(v-bmg.GAP)/bmg.GAP:11.4%}')

print()
print(f'GAP_IDENTITY stored in module: {bmg.GAP_IDENTITY!r}')
print(f'relative error of the identity: {abs(bmg.GAP_IDENTITY-bmg.GAP)/bmg.GAP:.4%}')

The 1/sqrt(2) factor is explained: the sigma=1/2 symmetry, the first
Cayley-Dickson doubling. **The 10^3 factor is an open question** and is recorded
as such — it is not derived anywhere in this repo.

In [ ]:
# The module's own validators. These print their own verdicts.
for fn in ['validate', 'gap_value', 'identity_check', 'bao_consistency',
           'mtheory_geometry']:
    print('=' * 60)
    print(fn)
    print('=' * 60)
    result = getattr(bmg, fn)()
    print(result)
    print()

## Where the gap sits

A log-scale view of how far apart the candidate expressions are. The point of
the plot is the 31.6x error, which is large enough to be obvious and is exactly
the mistake the README warns about.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
names = list(candidates) + ['GAP (computed)']
vals = [candidates[k] for k in candidates] + [bmg.GAP]
colours = ['#b04a3f', '#3f7fb0', '#3f7fb0', '#2f2f2f']
ax.barh(names, vals, color=colours)
ax.set_xscale('log')
ax.set_xlabel('value (log scale)')
ax.axvline(bmg.GAP, color='#2f2f2f', ls='--', lw=1)
ax.set_title('GAP vs candidate closed forms')
for i, v in enumerate(vals):
    ax.text(v * 1.1, i, f'{v:.3e}', va='center', fontsize=8)
plt.tight_layout(); plt.show()